# Bitcoin Price Prediction

**Tabular Regression Project** — Predict Bitcoin price from historical data.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Bitcoin Historical (367 rows × 2 columns)
Target: `Price`

## 2 · Project Overview

We predict Bitcoin's **Price** using a small historical dataset with only Date and Price columns. Since we have only one raw feature (Date), the key challenge is **feature engineering** — extracting year, month, day-of-week, and lag features to give models enough signal.

With only 367 rows this is a small-data problem where simpler models may compete with complex ones.

## 3 · Learning Objectives

1. Engineer temporal features from a date column.
2. Create lag and rolling-window features for price prediction.
3. Compare tree-based models on a tiny time-series-like dataset.
4. Understand limitations of tabular ML for financial time series.
5. Use LazyPredict and FLAML for rapid model comparison.

## 4 · Problem Statement

Given historical Bitcoin price data (Date and Price), predict future Bitcoin prices. This teaches feature engineering from minimal raw data and highlights why financial prediction is inherently difficult.

## 5 · Why This Project Matters

- Cryptocurrency price prediction is a popular ML application.
- Teaches how to extract predictive features from a single time column.
- Demonstrates that **financial markets are hard to predict** — an important lesson.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 367 |
| **Columns** | 2 |
| **Features** | Date |
| **Target** | Price (continuous, USD) |
| **File** | `bitcoin.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: Historical Bitcoin price data (public/educational use).
- **License**: Public domain.
- **Limitations**: Only ~1 year of data; no volume, market cap, or sentiment features.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Price'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'bitcoin.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min():.2f}, {df[TARGET].max():.2f}]')
print(f'Target mean: {df[TARGET].mean():.2f}')

## 13 · Exploratory Data Analysis

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df['Date'], df[TARGET])
axes[0].set_title('Bitcoin Price Over Time')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Price (USD)')

df[TARGET].hist(bins=30, ax=axes[1], edgecolor='black')
axes[1].set_title('Price Distribution')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(f'Price statistics:')
print(df[TARGET].describe())
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split Strategy

We use a random 80/20 split. For a production time-series model you would split by date, but here the goal is to demonstrate tabular regression techniques.

## 16 · Preprocessing Strategy

The raw dataset has only Date and Price. We need to engineer features from the Date column before we can train any model.

## 17 · Feature Engineering

We extract temporal features from Date and create lag/rolling features from Price.

In [ ]:
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day
df['dayofweek'] = df['Date'].dt.dayofweek
df['dayofyear'] = df['Date'].dt.dayofyear

# Lag features
for lag in [1, 3, 7, 14]:
    df[f'price_lag_{lag}'] = df[TARGET].shift(lag)

# Rolling features
for window in [7, 14, 30]:
    df[f'price_roll_mean_{window}'] = df[TARGET].shift(1).rolling(window).mean()
    df[f'price_roll_std_{window}'] = df[TARGET].shift(1).rolling(window).std()

df = df.dropna().reset_index(drop=True)
print(f'Shape after feature engineering: {df.shape}')
print(f'Features: {[c for c in df.columns if c not in [TARGET, "Date"]]}')

feature_cols = [c for c in df.columns if c not in [TARGET, 'Date']]
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)

print(f'Baseline Linear Regression:')
print(f'  RMSE: {baseline_rmse:.2f}')
print(f'  MAE:  {baseline_mae:.2f}')
print(f'  R²:   {baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=4,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=4,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=4,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=20, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Lag features** (especially 1-day lag) are the strongest predictors — Bitcoin price is highly autocorrelated.
- This is essentially learning momentum/trend continuation.
- **Caution**: High R² on random splits can be misleading for financial time series. In practice, predicting future prices from past prices alone is extremely difficult.
- This model should NOT be used for trading decisions.

## 26 · Limitations

- Only 367 data points with 2 original columns — very limited signal.
- Random split violates temporal ordering.
- No fundamental features (volume, market sentiment, macroeconomic indicators).
- Financial markets are notoriously efficient — past prices alone rarely predict future prices reliably.

## 27 · How to Improve This Project

1. Use a proper time-based train/test split.
2. Add volume, market cap, and on-chain metrics.
3. Incorporate sentiment data from social media.
4. Try LSTM or Transformer-based sequence models.
5. Add macro indicators (S&P 500, gold price, DXY).

## 28 · Production Considerations

- Financial prediction models require rigorous backtesting.
- Regulatory implications of automated trading advice.
- Model drift is extreme in crypto markets.
- Never deploy without extensive out-of-sample validation.

## 29 · Common Mistakes

1. **Look-ahead bias**: Using future data in features (lag features must use `.shift()`).
2. **Random splits on time series**: Inflates performance metrics.
3. **Overfitting on small data**: 367 rows with many lag features risks overfitting.
4. **Confusing correlation with causation**: High R² ≠ profitable trading strategy.

## 30 · Mini Challenge / Exercises

1. Implement a time-based 80/20 split and compare metrics.
2. Add a 'returns' feature (pct change) and see if it helps.
3. Try adding external data (S&P 500 index) as a feature.
4. Build a naive baseline (predict yesterday's price) and compare.

## 31 · Final Summary / Key Takeaways

- Feature engineering from a single Date column is the main challenge.
- Lag and rolling features capture autocorrelation effectively.
- Tree-based models perform well but beware of overfitting on small data.
- Financial prediction requires much more rigorous evaluation than standard ML.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline_linear_regression'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')